# 04 - Prompt Management：註冊、版本管理與自動優化

本筆記本示範如何使用 MLflow 3.x Prompt Registry 管理 prompt，
以及透過 Prompt Optimization 自動優化 prompt。

## 簡介

MLflow 3.x 提供完整的 Prompt 管理功能：

1. **Prompt Registry**：註冊、版本控制、搜尋 prompt
2. **Prompt Model Configuration**：prompt 可綁定模型設定
3. **Prompt Optimization**：使用 GEPA 演算法自動優化 prompt

本框架封裝了兩個模組：
- `app.prompts.PromptManager`：Prompt 註冊/載入/匯出
- `app.prompts.optimize_prompt`：Prompt 優化

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

import mlflow
from app.utils.config import init_config
from app.logger import init_mlflow
from app.prompts import PromptManager

cfg = init_config()
init_mlflow(cfg)

print("設定與 MLflow 初始化完成。")

## Prompt 註冊與載入

### 使用 PromptManager（框架封裝）

`PromptManager` 提供 MLflow-first + local fallback 的 prompt 管理。

In [ ]:
pm = PromptManager(cfg)

# 註冊 prompt（使用 {{ variable }} 語法）
version = pm.register(
    name="qa_prompt",
    template="請回答以下問題：{{ question }}\n\n背景資訊：{{ context }}",
    commit_message="Initial QA prompt",
)
print(f"Registered version: {version}")

# 載入 prompt
prompt = pm.load("qa_prompt")
print(f"\nLoaded prompt type: {type(prompt).__name__}")
if hasattr(prompt, 'template'):
    print(f"Template: {prompt.template}")
else:
    print(f"Template: {prompt}")

In [ ]:
# 載入並格式化 prompt
formatted = pm.load_and_format(
    "qa_prompt",
    question="什麼是 MLflow？",
    context="MLflow 是一個開源 ML 生命週期管理平台。",
)
print("=== 格式化後的 Prompt ===")
print(formatted)

### 直接使用 MLflow GenAI API

也可以直接使用 `mlflow.genai` API 管理 prompt。

In [ ]:
# 直接使用 MLflow API 註冊 prompt（含 model_config）
prompt_v = mlflow.genai.register_prompt(
    name="summarize_prompt",
    template="請用 {{ language }} 摘要以下內容（最多 {{ max_words }} 字）：\n\n{{ text }}",
    commit_message="Summarize prompt with model config",
    model_config={
        "model": "gpt-4o-mini",
        "temperature": 0.3,
        "max_tokens": 500,
    },
)
print(f"Registered: {prompt_v.name} v{prompt_v.version}")

# 載入最新版本
loaded = mlflow.genai.load_prompt("prompts:/summarize_prompt@latest")
print(f"\nTemplate: {loaded.template}")

# 格式化
result = loaded.format(
    language="繁體中文",
    max_words="100",
    text="MLflow is an open-source platform for managing the end-to-end ML lifecycle.",
)
print(f"\nFormatted:\n{result}")

## 搜尋 Prompt

使用 `mlflow.genai.search_prompts()` 搜尋已註冊的 prompt。

In [ ]:
# 搜尋所有 prompt
prompts = mlflow.genai.search_prompts()
print(f"共有 {len(prompts)} 個已註冊的 prompt：")
for p in prompts:
    print(f"  - {p.name}")

# 搜尋特定名稱
# matched = mlflow.genai.search_prompts(filter_string="name LIKE '%qa%'")

## 匯出 Prompt 為 Local 檔案

部署環境不需依賴 MLflow server，可將 prompt 匯出為本地 markdown 檔案。

In [ ]:
# 匯出所有 prompt 到本地目錄
exported = pm.export_all("../prompts")
print(f"匯出了 {len(exported)} 個 prompt：")
for path in exported:
    print(f"  {path}")

## Prompt Optimization（自動優化）

使用 MLflow 3.5+ 的 `optimize_prompts()` API，基於 GEPA 演算法自動搜尋最佳 prompt 版本。

**關鍵要求**：`predict_fn` 內部必須使用 `mlflow.genai.load_prompt()` 載入 prompt，
optimizer 才能自動替換不同版本。

> **注意**：Prompt Optimization 需要 LLM API key（用於生成候選 prompt 和評估）。

In [ ]:
from mlflow.genai.scorers import scorer


# 1. 註冊要優化的 prompt
pm.register(
    name="math_prompt",
    template="{{ question }}",
    commit_message="Initial math prompt for optimization",
)


# 2. 定義 predict_fn（必須透過 load_prompt 載入 prompt）
@mlflow.trace
def math_predict(question: str) -> str:
    prompt = mlflow.genai.load_prompt("prompts:/math_prompt@latest")
    formatted = prompt.format(question=question)
    # 實際應呼叫 LLM，此處用 mock
    return formatted  # placeholder


# 3. 定義 scorer
@scorer
def math_exact_match(outputs, expectations):
    expected = expectations.get("answer", "")
    return 1.0 if outputs.strip() == str(expected).strip() else 0.0


# 4. 準備訓練資料
train_data = [
    {"inputs": {"question": "What is 2+2?"}, "expectations": {"answer": "4"}},
    {"inputs": {"question": "What is 3*5?"}, "expectations": {"answer": "15"}},
    {"inputs": {"question": "What is 10-7?"}, "expectations": {"answer": "3"}},
    {"inputs": {"question": "What is 8/2?"}, "expectations": {"answer": "4"}},
]

print(f"準備了 {len(train_data)} 筆訓練資料。")
print("\n注意：執行下方的 optimize 需要 LLM API key。")

In [ ]:
# 5. 執行 Prompt Optimization
from app.prompts import optimize_prompt

try:
    result = optimize_prompt(
        predict_fn=math_predict,
        train_data=train_data,
        prompt_name="math_prompt",
        scorers=[math_exact_match],
        reflection_model="openai:/gpt-4o",
    )

    print("=== Prompt Optimization 結果 ===")
    print(f"優化後的 prompt URI: {result.prompt.uri}")
    print(f"優化後的 template:\n{result.prompt.template}")

except Exception as e:
    print(f"[提示] Prompt Optimization 需要 LLM API key：{e}")
    print("請設定 OPENAI_API_KEY 環境變數後重試。")

## 小結

| 功能 | API | 說明 |
|------|-----|------|
| 註冊 Prompt | `PromptManager.register()` / `mlflow.genai.register_prompt()` | 版本控制 + model config |
| 載入 Prompt | `PromptManager.load()` / `mlflow.genai.load_prompt()` | 載入特定版本 |
| 格式化 | `PromptManager.load_and_format()` / `prompt.format()` | 變數替換 |
| 搜尋 | `mlflow.genai.search_prompts()` | 搜尋已註冊 prompt |
| 匯出 | `PromptManager.export_all()` | 匯出為本地 markdown |
| 優化 | `optimize_prompt()` / `mlflow.genai.optimize_prompts()` | GEPA 自動優化 |

### 建議工作流程

1. 在 Prompt Registry 註冊初始版本
2. 使用 `evaluate()` 建立 baseline 指標
3. 使用 `optimize_prompts()` 自動搜尋更好的版本
4. 在 MLflow UI 比較不同版本的 evaluation 結果
5. 部署前匯出 prompt 為 local 檔案